In [16]:
%pip list

Package                   Version
------------------------- -----------
affine                    3.0.0
anyio                     4.14.2
appnope                   1.0.0
argon2-cffi               25.1.0
argon2-cffi-bindings      26.1.0
arrow                     1.4.0
arviz                     1.3.0
arviz-base                1.3.0
arviz-plots               1.3.1
arviz-stats               1.3.1
asttokens                 3.0.2
async-lru                 2.3.0
attrs                     26.1.0
babel                     2.18.0
bambi                     0.20.0
beautifulsoup4            4.15.0
bleach                    6.4.0
cachetools                7.1.7
certifi                   2026.7.22
cffi                      2.1.1
charset-normalizer        3.5.1
click                     8.4.2
click-plugins             1.1.1.2
cligj                     0.7.2
cloudpickle               3.1.2
comm                      0.2.3
cons                      0.4.7
contourpy                 1.3.3
cycler             

In [7]:
import pandas as pd

In [8]:
raw_df = pd.read_csv("events.csv")

print("Raw rows:", len(raw_df))
raw_df.head()

Raw rows: 432


,event_id,event_timestamp,customer_id,product_id,event_type,quantity,unit_price,country,source,status
0,E000157,2026-08-13 13:06:46,C00059,P0027,purchase,2,110.37,MX,mobile,processed
1,E000021,2026-08-13 21:51:05,C00125,P0053,view,1,25.99,US,partner_api,ok
2,E000356,2026-08-08 22:22:55,C00113,P0056,purchase,1,205.10,US,web,ok
3,E000006,2026-08-01 23:43:46,C00032,P0060,view,1,117.56,GB,web,failed
4,E000124,2026-08-12 02:29:55,C00018,P0016,purchase,5,13.80,MX,web,failed


In [9]:
def remove_exact_duplicates(df):
    """
    Remove exact duplicate rows from the dataset.

    Returns:
        cleaned DataFrame
        number of duplicates removed
    """
    df = df.copy()

    duplicates_removed = df.duplicated().sum()
    df = df.drop_duplicates()

    return df, duplicates_removed

In [15]:
df.duplicated?
# or help(df.duplicated)

Signature:
df.duplicated(
    subset: 'Hashable | Iterable[Hashable] | None' = None,
    keep: 'DropKeep' = 'first',
) -> 'Series'
Docstring:
Return boolean Series denoting duplicate rows.

Considering certain columns is optional.

Parameters
----------
subset : column label or iterable of labels, optional
    Only consider certain columns for identifying duplicates, by
    default use all of the columns.
keep : {'first', 'last', False}, default 'first'
    Determines which duplicates (if any) to mark.

    - ``first`` : Mark duplicates as ``True`` except for the first occurrence.
    - ``last`` : Mark duplicates as ``True`` except for the last occurrence.
    - False : Mark all duplicates as ``True``.

Returns
-------
Series
    Boolean series for each duplicated rows.

See Also
--------
Index.duplicated : Equivalent method on index.
Series.duplicated : Equivalent method on Series.
Series.drop_duplicates : Remove duplicate values from Series.
DataFrame.drop_duplicates : Remove duplica

In [10]:
def standardize_text(df):
    """
    Standardize event_type and status by:
    - removing leading/trailing whitespace
    - converting text to lowercase
    """
    df = df.copy()

    df["event_type"] = (
        df["event_type"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    df["status"] = (
        df["status"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    return df

In [11]:
def convert_types_and_check(df):
    """
    Convert timestamp and numeric columns to appropriate data types.

    Invalid timestamps or numbers are converted to missing values
    instead of causing the pipeline to fail.

    Also identifies missing customer/product IDs.
    """
    df = df.copy()

    # Clean identifier columns so blank strings count as missing
    for col in ["event_id", "customer_id", "product_id"]:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

    # Save original columns so we can identify invalid values
    original_timestamp = df["event_timestamp"].copy()
    original_quantity = df["quantity"].copy()
    original_price = df["unit_price"].copy()

    # Convert timestamp
    df["event_timestamp"] = pd.to_datetime(
        df["event_timestamp"],
        errors="coerce"
    )

    # Convert quantity to numeric
    quantity = pd.to_numeric(
        df["quantity"],
        errors="coerce"
    )

    # Quantity represents units, so non-whole numbers are invalid
    non_integer_quantity = quantity.notna() & (quantity % 1 != 0)
    quantity = quantity.mask(non_integer_quantity)

    df["quantity"] = quantity.astype("Int64")

    # Convert unit_price to numeric
    df["unit_price"] = pd.to_numeric(
        df["unit_price"],
        errors="coerce"
    )

    # Identify values that were present but could not be converted
    invalid_timestamp = (
        original_timestamp.notna()
        & df["event_timestamp"].isna()
    )

    invalid_quantity = (
        original_quantity.notna()
        & df["quantity"].isna()
    )

    invalid_price = (
        original_price.notna()
        & df["unit_price"].isna()
    )

    quality_report = {
        "invalid timestamps": int(invalid_timestamp.sum()),
        "invalid quantities": int(invalid_quantity.sum()),
        "invalid unit prices": int(invalid_price.sum()),
        "missing event IDs": int(df["event_id"].isna().sum()),
        "missing customer IDs": int(df["customer_id"].isna().sum()),
        "missing product IDs": int(df["product_id"].isna().sum())
    }

    return df, quality_report

In [12]:
# Function 1
df, duplicates_removed = remove_exact_duplicates(raw_df)

# Function 2
df = standardize_text(df)

# Function 3
clean_df, quality_report = convert_types_and_check(df)

print("Duplicates removed:", duplicates_removed)
print("\nData-quality issues:")

for issue, count in quality_report.items():
    print(f"{issue}: {count}")

Duplicates removed: 12

Data-quality issues:
invalid timestamps: 8
invalid quantities: 4
invalid unit prices: 2
missing event IDs: 0
missing customer IDs: 16
missing product IDs: 8


In [13]:
clean_df.head()

,event_id,event_timestamp,customer_id,product_id,event_type,quantity,unit_price,country,source,status
0,E000157,2026-08-13 13:06:46,C00059,P0027,purchase,2,110.37,MX,mobile,processed
1,E000021,2026-08-13 21:51:05,C00125,P0053,view,1,25.99,US,partner_api,ok
2,E000356,2026-08-08 22:22:55,C00113,P0056,purchase,1,205.10,US,web,ok
3,E000006,2026-08-01 23:43:46,C00032,P0060,view,1,117.56,GB,web,failed
4,E000124,2026-08-12 02:29:55,C00018,P0016,purchase,5,13.80,MX,web,failed


### Task 1: Data Cleaning

I created three reusable cleaning functions. The first function removes
exact duplicate rows. The second standardizes `event_type` and `status`
by stripping whitespace and converting the values to lowercase. The
third function converts timestamps and numeric columns to appropriate
data types and uses `errors="coerce"` so malformed values become missing
values instead of stopping the pipeline. It also reports invalid values
and missing identifiers.

In [18]:
def calculate_revenue(df):
    df["revenue"] = df["quantity"] * df[" unit_price "]
    purchases = df[df[" event_type "] == "Purchase"]
    return purchases.groupby("country")["reveneu"].sum()

In [21]:
# %xmode Context
%xmode Verbose

Exception reporting mode: Verbose


In [22]:
calculate_revenue(clean_df)

KeyError: ' unit_price '

In [23]:
# Corrected function
def calculate_revenue(df):
    """
    Calculate total purchase revenue by country.
    """
    df = df.copy()

    df["revenue"] = df["quantity"] * df["unit_price"]

    purchases = df[df["event_type"] == "purchase"]

    return (
        purchases.groupby("country")["revenue"]
        .sum()
        .sort_values(ascending=False)
    )


# Run corrected function
purchase_revenue = calculate_revenue(clean_df)

purchase_revenue

country
DE    9732.48
GB    8692.54
MX    6955.25
CA    6654.89
US     3547.6
Name: revenue, dtype: Float64

In [ ]:
### Task 2: Debugging the Revenue Transformation

The traceback showed a `KeyError`, which indicated that the function was
trying to access a column that did not exist. I inspected the column names
and found that the original function contained extra spaces around
`unit_price` and `event_type`. I also found that `reveneu` was misspelled
and that the cleaned `event_type` values were lowercase, so `"Purchase"`
needed to be changed to `"purchase"`.

I used `%xmode Verbose` and `%debug` to inspect the error and the available
DataFrame columns. After correcting these issues, the function successfully
calculated purchase revenue by country.